# KissanSevaAI · Phase 1 rebuild (DINOv2-S + FAISS + ONNX)

**Goal:** Rebuild crop-disease retrieval from the original labeled images, select k-NN settings using a validation split, evaluate a held-out test split, export a **compatible** ONNX model and FAISS index for a FastAPI server.

**Hardware:** Colab T4. **Important:** This is a frozen-embedding pipeline, **not** DINOv2 fine-tuning. Run all cells in order. Use `pilot` mode before `full`.

**Evaluation limitation:** Exact-file duplicate removal is implemented, but near-duplicate photos of the same leaf may remain. Before claiming production accuracy, additionally test on independent real-farm images, include non-leaf images, and review the licensing of the source dataset.

## 1. Install and mount Drive

In [ ]:
!pip -q install faiss-cpu scikit-learn pandas pillow tqdm onnx onnxruntime

In [ ]:
import os, sys, json, time, zipfile, hashlib, random, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch, faiss
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive

drive.mount('/content/drive')
print('CUDA:', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU'
random.seed(42); np.random.seed(42); torch.manual_seed(42)
DEVICE = 'cuda'
MODE = 'pilot'               # 'pilot' first; change to 'full' and rerun from this cell
ZIP_PATH = Path('/content/drive/MyDrive/archive.zip')  # edit if needed
WORK = Path('/content/plant_disease_dataset')
OUT = Path('/content/drive/MyDrive/KissanSevaAI_Phase1') / MODE
OUT.mkdir(parents=True, exist_ok=True)
BATCH = 24                  # Reduce to 8 if GPU out-of-memory
CHUNK = 480                 # Each embedding chunk saved to Drive
SIZE = 224                 # divisible by DINOv2 patch size 14
assert ZIP_PATH.is_file(), f'Cannot find {ZIP_PATH}'
assert zipfile.is_zipfile(ZIP_PATH), 'Dataset path is not a ZIP archive'
print('MODE:', MODE, '| OUTPUT:', OUT)

## 2. Extract archive and locate crop/disease/image folders

In [ ]:
DONE = WORK / '.extract_done'
if not DONE.exists():
    if WORK.exists(): shutil.rmtree(WORK)
    WORK.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        # Reject paths outside destination before extracting.
        base = WORK.resolve()
        for m in z.infolist():
            if not (base == (base / m.filename).resolve() or base in (base / m.filename).resolve().parents):
                raise ValueError('Unsafe filename in archive: ' + m.filename)
        z.extractall(WORK)
    DONE.touch()

roots = [p for p in WORK.rglob('crops') if p.is_dir()]
assert roots, 'Cannot locate crops/ folder inside extracted ZIP'
CROPS = min(roots, key=lambda p: len(p.parts))
print('Crop folder:', CROPS)
print('Example crops:', [p.name for p in CROPS.iterdir() if p.is_dir()][:15])

## 3. Build manifest and remove exact duplicates

In [ ]:
EXT = {'.jpg','.jpeg','.png','.webp','.bmp'}
MANIFEST = OUT / 'manifest.csv'
if MANIFEST.exists():
    df = pd.read_csv(MANIFEST)
else:
    candidates = []
    for crop in sorted(CROPS.iterdir()):
        if not crop.is_dir(): continue
        for condition in sorted(crop.iterdir()):
            if not condition.is_dir(): continue
            label = f'{crop.name}/{condition.name}'
            for img in sorted(condition.rglob('*')):
                if img.is_file() and img.suffix.lower() in EXT:
                    candidates.append((str(img.relative_to(CROPS)), label))
    assert candidates, 'No images found; inspect your ZIP folder structure'
    seen = {} ; rows=[] ; conflicts=[]
    for rel,label in tqdm(candidates, desc='Hashing images'):
        h = hashlib.sha256()
        with open(CROPS/rel,'rb') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
        dig = h.hexdigest()
        if dig in seen:
            if seen[dig] != label: conflicts.append(dig)
            continue
        seen[dig] = label
        rows.append({'path':rel, 'label':label, 'sha256':dig})
    df = pd.DataFrame(rows)
    # Drop ambiguous duplicates seen under multiple labels.
    if conflicts: df = df[~df.sha256.isin(set(conflicts))].reset_index(drop=True)
    if MODE == 'pilot':
        df = (df.groupby('label', group_keys=False)
                .apply(lambda group: group.sample(n=min(40,len(group)), random_state=42))
                .reset_index(drop=True))
    df = df.sort_values(['label','path']).reset_index(drop=True)
    df.to_csv(MANIFEST,index=False)
assert df['label'].nunique() > 1
counts = df.label.value_counts()
assert counts.min() >= 10, f'Insufficient per-class images: min={counts.min()}'
print('MODE',MODE,'images',len(df),'classes',len(counts),'minimum/class',counts.min())

## 4. Deterministic 80/10/10 train/validation/test split

In [ ]:
SPLITS = OUT / 'splits.csv'
if SPLITS.exists():
    split_df = pd.read_csv(SPLITS)
    assert len(split_df) == len(df) and set(split_df.sha256)==set(df.sha256), 'Existing split does not match dataset'
else:
    indices = np.arange(len(df))
    train_i, other_i = train_test_split(indices, test_size=.2, stratify=df.label, random_state=42)
    val_i, test_i = train_test_split(other_i, test_size=.5, stratify=df.label.iloc[other_i],random_state=42)
    split_df = df.copy(); split_df['split'] = ''
    split_df.loc[train_i,'split']='train'; split_df.loc[val_i,'split']='val'; split_df.loc[test_i,'split']='test'
    split_df.to_csv(SPLITS,index=False)
classes = sorted(split_df.label.unique().tolist())
class_to_id = {c:i for i,c in enumerate(classes)}
(OUT/'classes.json').write_text(json.dumps(classes,indent=2))
print(split_df.split.value_counts().to_dict(), 'classes:',len(classes))

## 5. Load pretrained DINOv2-S/14, the shared embedding definition, and preprocessing

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
MEAN=[.485,.456,.406]; STD=[.229,.224,.225]
PREPROCESS=transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(SIZE), transforms.ToTensor(),
    transforms.Normalize(MEAN,STD)
])
# ViT-S = 384 dimensional CLS token; concatenate CLS + average patch token => 768.
backbone = torch.hub.load('facebookresearch/dinov2','dinov2_vits14',trust_repo=True).eval().to(DEVICE)
backbone.requires_grad_(False)

class DinoFeatures(nn.Module):
    def __init__(self, encoder):
        super().__init__(); self.encoder = encoder
    def forward(self,x):
        feat=self.encoder.forward_features(x)
        cls=feat['x_norm_clstoken']; avg=feat['x_norm_patchtokens'].mean(dim=1)
        return F.normalize(torch.cat([cls,avg],dim=1),dim=1)

model=DinoFeatures(backbone).eval().to(DEVICE)
with torch.inference_mode():
    out_shape=model(torch.zeros(1,3,SIZE,SIZE,device=DEVICE)).shape
assert tuple(out_shape)==(1,768),f'Unexpected embedding shape: {out_shape}'
print('Feature encoder OK:',out_shape)

## 6. Generate and cache embeddings (resumes by chunk after Colab disconnect)

In [ ]:
class ImgDataset(Dataset):
    def __init__(self, frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        p=CROPS/self.frame.iloc[i]['path']
        with Image.open(p) as raw:
            img=ImageOps.exif_transpose(raw).convert('RGB')
            x=PREPROCESS(img)
        return x

def get_features(part):
    part_df = split_df[split_df.split==part].reset_index(drop=True)
    cache=OUT/'embedding_chunks'/part
    cache.mkdir(parents=True,exist_ok=True)
    gathered=[]
    for start in tqdm(range(0,len(part_df),CHUNK),desc=f'Embeddings: {part}'):
        stop=min(start+CHUNK,len(part_df))
        dest=cache/f'{start:07d}_{stop:07d}.npy'
        if dest.exists():
            arr=np.load(dest)
            assert arr.shape == (stop-start,768),f'Bad existing chunk: {dest}'
        else:
            ds=ImgDataset(part_df.iloc[start:stop])
            dl=DataLoader(ds,batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=True)
            chunks=[]
            with torch.inference_mode():
                for imgs in dl:
                    with torch.autocast('cuda',dtype=torch.float16):
                        emb=model(imgs.to(DEVICE)).float().cpu().numpy()
                    chunks.append(emb)
            arr=np.concatenate(chunks).astype('float32')
            arr/=np.maximum(np.linalg.norm(arr,axis=1,keepdims=True),1e-9)
            tmp=dest.with_suffix('.tmp.npy')
            np.save(tmp,arr)
            os.replace(tmp,dest)
        gathered.append(arr)
    X=np.concatenate(gathered).astype('float32')
    y=np.array([class_to_id[x] for x in part_df.label],dtype=np.int32)
    assert np.isfinite(X).all() and len(X)==len(y) and X.shape[1]==768
    return X,y,part_df

X_train,y_train,train_df = get_features('train')
X_val,y_val,val_df = get_features('val')
X_test,y_test,test_df = get_features('test')
print('Embeddings:',X_train.shape,X_val.shape,X_test.shape)

## 7. Build FAISS cosine index and select k by validation only

In [ ]:
D=768
# Exact flat index for pilot; IVF speeds up large CPU searches.
if len(X_train)<10000:
    index=faiss.IndexFlatIP(D)
else:
    nlist=256
    index=faiss.IndexIVFFlat(faiss.IndexFlatIP(D),D,nlist,faiss.METRIC_INNER_PRODUCT)
    rng=np.random.default_rng(42)
    subset=X_train[rng.choice(len(X_train),size=min(len(X_train),30000),replace=False)]
    index.train(subset)
    index.nprobe=64
index.add(X_train)
assert index.ntotal==len(y_train)

def predict_knn(sims,positions,k,method):
    predictions=[]; agreements=[]
    for ss,ii in zip(sims[:,:k],positions[:,:k]):
        good=ii>=0
        ids=y_train[ii[good]]; values=ss[good]
        if len(ids)==0:
            predictions.append(-1); agreements.append(0.); continue
        weights=np.ones(len(ids)) if method=='majority' else np.exp(10*(values-values.max()))
        tallies=np.bincount(ids, weights=weights, minlength=len(classes))
        label=int(np.argmax(tallies))
        predictions.append(label)
        agreements.append(float(tallies[label]/max(tallies.sum(),1e-8)))
    return np.array(predictions),np.array(agreements)

K_GRID=[1,3,5,9,15]
KMAX=min(max(K_GRID),len(X_train))
val_sims,val_ids=index.search(X_val,KMAX)
rows=[]
for k in K_GRID:
    if k>KMAX:continue
    for method in ['majority','weighted']:
        pred, agree=predict_knn(val_sims,val_ids,k,method)
        rows.append({'k':k,'method':method,
                     'accuracy':accuracy_score(y_val,pred),
                     'macro_f1':f1_score(y_val,pred,average='macro',zero_division=0)})
results=pd.DataFrame(rows).sort_values(['macro_f1','accuracy'],ascending=False)
print(results.to_string(index=False))
results.to_csv(OUT/'validation_knn.csv',index=False)
choice=results.iloc[0]
BEST_K=int(choice['k']); BEST_METHOD=str(choice['method'])
print('Selected using validation:',BEST_K,BEST_METHOD)

## 8. Evaluate once on held-out test (do not tune on test results)

In [ ]:
test_sims,test_ids=index.search(X_test,KMAX)
test_pred,test_agree=predict_knn(test_sims,test_ids,BEST_K,BEST_METHOD)
acc=float(accuracy_score(y_test,test_pred))
f1=float(f1_score(y_test,test_pred,average='macro',zero_division=0))
print(f'HELD-OUT TEST: accuracy={acc:.4f}, macro-F1={f1:.4f}')
report=classification_report(y_test,test_pred,labels=np.arange(len(classes)),target_names=classes,output_dict=True,zero_division=0)
pd.DataFrame(report).T.to_csv(OUT/'test_classification_report.csv')
(OUT/'test_metrics.json').write_text(json.dumps({'accuracy':acc,'macro_f1':f1,'n_test':len(y_test)},indent=2))
# No statistically calibrated probability or out-of-distribution detector exists here.
# Agreement below 0.60 flags low support, not a medically validated threshold.
print('Low neighbor-agreement rate:',float((test_agree<.6).mean()))

## 9. Export the **same** feature definition to ONNX and compare with PyTorch

In [ ]:
import onnxruntime as ort
onnx_path=OUT/'dinov2_vits14_cls_mean.onnx'
dummy=torch.randn(1,3,SIZE,SIZE,device=DEVICE)
model.eval()
with torch.inference_mode():
    torch.onnx.export(model,dummy,str(onnx_path),input_names=['image'],output_names=['embedding'],
                      dynamic_axes={'image':{0:'batch'},'embedding':{0:'batch'}},
                      opset_version=17,do_constant_folding=True,dynamo=False)
sess=ort.InferenceSession(str(onnx_path),providers=['CPUExecutionProvider'])
with torch.inference_mode(): ref=model(dummy).float().cpu().numpy()
onx=sess.run(None,{'image':dummy.detach().cpu().numpy()})[0]
assert onx.shape==(1,D),f'Wrong ONNX output shape: {onx.shape}'
max_diff=float(np.abs(ref-onx).max());print('ONNX export OK. max abs diff:',max_diff)
assert max_diff<0.02,'ONNX and PyTorch feature implementations differ; STOP and fix before integration'

## 10. Package model, vectors and metadata for the API

In [ ]:
faiss_path=OUT/'crop_index.faiss'
faiss.write_index(index,str(faiss_path))
np.save(OUT/'train_labels.npy',y_train)
train_df[['path','label']].to_csv(OUT/'train_image_mapping.csv',index=False)
metadata={
    'model':'dinov2_vits14', 'features':'normalized_concat_cls_and_mean_patch',
    'embedding_dim':D,'image_size':SIZE,'resize_short_side':256,
    'mean':MEAN,'std':STD, 'class_count':len(classes),
    'k':BEST_K,'vote_method':BEST_METHOD,
    'cosine_metric':'IndexFlatIP' if len(X_train)<10000 else 'IndexIVFFlat_IP',
    'nprobe':int(index.nprobe) if hasattr(index,'nprobe') else None,
    'low_agreement_threshold':0.60,
    'warning':'Similarity and neighbor agreement are NOT calibrated disease probabilities; field testing required.',
    'validation_macro_f1':float(choice['macro_f1']),
    'test_accuracy':acc,'test_macro_f1':f1
}
(OUT/'model_meta.json').write_text(json.dumps(metadata,indent=2))
assert len(y_train)==index.ntotal and len(classes)==metadata['class_count']
print('Ready for server:')
for f in ['dinov2_vits14_cls_mean.onnx','crop_index.faiss','train_labels.npy','classes.json','model_meta.json']:
    p=OUT/f; print(f,'=>',round(p.stat().st_size/1024**2,1),'MB')
print('Output:',OUT)

## 11. Sanity check: replay one held-out test photo through ONNX and FAISS

In [ ]:
pos=0
example=test_df.iloc[pos]
with Image.open(CROPS/example['path']) as raw:
    image=ImageOps.exif_transpose(raw).convert('RGB')
input_np=PREPROCESS(image).unsqueeze(0).numpy().astype('float32')
q=sess.run(None,{'image':input_np})[0].astype('float32')
faiss.normalize_L2(q)
# The same image should embed almost identically through PyTorch and ONNX.
feature_cosine=float(np.dot(q[0],X_test[pos]))
print('ONNX ↔ extraction cosine:',feature_cosine)
assert feature_cosine>0.995, 'ONNX/preprocessing does not match the saved vectors; do not integrate.'
s,i=index.search(q,BEST_K)
pr,ag=predict_knn(s,i,BEST_K,BEST_METHOD)
print('Actual:',example['label'])
print('Predicted:',classes[int(pr[0])])
print('Top cosine:',float(s[0,0]),'| neighbor agreement:',float(ag[0]))
print('This sample is from the held-out split, NOT the FAISS training index.')

## 12. Deploy and integrate

Download **these five files** from your Drive output folder and copy them into `backend/model_assets/`:

- `dinov2_vits14_cls_mean.onnx`
- `crop_index.faiss`
- `train_labels.npy`
- `classes.json`
- `model_meta.json`

Use the accompanying FastAPI server and Expo client from the downloadable ZIP package. **Do not deploy** until the API test returns the correct output shape, the independent test metrics are acceptable, and non-leaf/uncertain inputs are handled sensibly. For higher accuracy, test larger DINOv2 models and independent real-world photographs. These features are NOT a veterinary/agronomic diagnosis.